# Colab GPU pipeline — Final Clean Generalization Protocol

This notebook implements the final experiment design:

1. Create one deterministic train/val/test split from `full_curated_v1`, stratified by `subset`.
2. Train YOLO/RT-DETR on the **training split containing all three training conditions**:
   - `synthetic_drone_positive`
   - `synthetic_distractor_only` / hard negatives
   - `synthetic_drone_plus_distractor` / mixed
3. Evaluate every model separately on the held-out test subsets:
   - `test_drone_positive`
   - `test_hard_negative`
   - `test_mixed`
4. Evaluate GroundingDINO zero-shot on the same test subsets.
5. Save one combined table with `eval_subset` and model metrics.

Important: the hard-negative subset is used as negative training data together with positive drone examples. We do **not** train a standalone detector only on hard negatives.


In [1]:
# 1) Configuration, Drive mount, GPU check, helpers
from pathlib import Path
import os
import sys
import subprocess
import shutil
from datetime import datetime

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
except Exception as e:
    print("Drive mount failed or not running in Colab:", repr(e))

# Your current Drive project path
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification"

PROJECT_ROOT = Path(DRIVE_PROJECT_PATH)
DATASET_ROOT = PROJECT_ROOT / "data/synthetic/full_curated_v1"

# Final clean generalization protocol:
# Train on train split containing positives + hard negatives + mixed.
# Evaluate separately on held-out test_drone_positive, test_hard_negative, test_mixed.
RUN_MODE = "final_clean"

# For final clean, use all split rows by default.
# Set TRAIN_MAX_IMAGES to an integer only for quick debugging.
TRAIN_MAX_IMAGES = None
EVAL_MAX_IMAGES_PER_SUBSET = None

EPOCHS_YOLO = 30
EPOCHS_RTDETR = 5

SPLIT_NAME = "full_curated_v1_final_clean_split"
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15

BATCH_YOLO = 16
BATCH_RTDETR = 2  # safer for L4
IMGSZ = 640

BASE_EVAL_CONFIG = "configs/eval_colab_full_curated_v1.yaml"
EVAL_CONFIG_DIR = PROJECT_ROOT / "configs/generated_eval_subsets"
EVAL_OUTPUT_ROOT = PROJECT_ROOT / "outputs/evaluation/final_clean_full_curated_v1"

# Stable run names. The eval config should point to these stable folders.
YOLO_RUN_NAME = "yolo11n_drone_colab"
RTDETR_RUN_NAME = "rtdetr_l_drone_colab"

RUN_GDINO = True
RUN_YOLO_TRAINING = True
RUN_RTDETR_TRAINING = True
RUN_YOLO_INFERENCE = True
RUN_RTDETR_INFERENCE = True

# Set True only when you intentionally want to delete previous training/prediction outputs.
RESET_MODEL_OUTPUTS = False
RESET_PREDICTIONS = True

# Evaluation subsets derived from held-out test rows.
EVAL_SUBSETS = {
    "test_drone_positive": "synthetic_drone_positive",
    "test_hard_negative": "synthetic_distractor_only",
    "test_mixed": "synthetic_drone_plus_distractor",
}

def list_training_runs(root, pattern):
    """Return real Ultralytics run folders that contain weights/best.pt.

    Excludes backup folders created by this notebook, because their names can also match
    patterns like `yolo11n_drone_colab*` and can otherwise be accidentally selected
    as the latest run.
    """
    root = Path(root)
    runs = []
    for p in root.glob(pattern):
        if not p.is_dir():
            continue
        name = p.name.lower()
        if "backup" in name or "old_before_latest_sync" in name or "disabled" in name:
            continue
        if (p / "weights/best.pt").exists():
            runs.append(p)
    return sorted(runs)

def get_latest_training_run(root, pattern):
    """Return the latest real Ultralytics run folder by modification time."""
    runs = list_training_runs(root, pattern)
    if not runs:
        return None
    return max(runs, key=lambda p: p.stat().st_mtime)

def sync_latest_run_to_stable(root, pattern, stable_name):
    """Copy latest Ultralytics run, including suffix folders like -2/-3, to stable_name.

    Why this is needed:
    Ultralytics avoids overwriting existing runs. If `name=yolo11n_drone_colab` already exists,
    a new training run may be saved as `yolo11n_drone_colab-2`, `-3`, etc.
    The evaluation config points to the stable folder `yolo11n_drone_colab`, so this sync step
    prevents evaluation from accidentally using an old checkpoint.
    """
    root = Path(root)
    latest = get_latest_training_run(root, pattern)
    if latest is None:
        print(f"No training runs found for {pattern} under {root}")
        return None

    stable = root / stable_name
    if latest.resolve() == stable.resolve():
        print(f"Stable folder is already latest: {stable}")
        return stable

    backup = root / f"{stable_name}_old_before_latest_sync"
    if backup.exists():
        shutil.rmtree(backup)
    if stable.exists():
        shutil.move(str(stable), str(backup))
        print(f"Backed up old stable folder to: {backup}")

    shutil.copytree(latest, stable)
    print(f"Synced latest run to stable folder:")
    print(f"  latest: {latest}")
    print(f"  stable: {stable}")
    return stable

def format_cmd(cmd):
    return " ".join([repr(str(x)) if " " in str(x) else str(x) for x in cmd])

def run_command(cmd, check=True, cwd=None):
    cmd = [str(x) for x in cmd]
    print("\nRunning:")
    print(format_cmd(cmd))
    result = subprocess.run(
        cmd,
        text=True,
        capture_output=True,
        cwd=str(cwd) if cwd is not None else None,
    )
    print("\n===== STDOUT =====")
    print(result.stdout)
    print("\n===== STDERR =====")
    print(result.stderr)
    print("\nReturn code:", result.returncode)
    if check and result.returncode != 0:
        raise RuntimeError("Command failed. See STDOUT/STDERR above.")
    return result

print("CUDA check:")
try:
    import torch
    print("CUDA:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch check failed:", repr(e))

print("\nRun configuration:")
print("RUN_MODE:", RUN_MODE)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_ROOT:", DATASET_ROOT)
print("SPLIT_NAME:", SPLIT_NAME)
print("TRAIN_MAX_IMAGES:", TRAIN_MAX_IMAGES)
print("EVAL_MAX_IMAGES_PER_SUBSET:", EVAL_MAX_IMAGES_PER_SUBSET)
print("EPOCHS_YOLO:", EPOCHS_YOLO)
print("EPOCHS_RTDETR:", EPOCHS_RTDETR)
print("EVAL_OUTPUT_ROOT:", EVAL_OUTPUT_ROOT)


Mounted at /content/drive
CUDA check:
CUDA: True
GPU: NVIDIA L4

Run configuration:
RUN_MODE: final_clean
PROJECT_ROOT: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification
DATASET_ROOT: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/synthetic/full_curated_v1
SPLIT_NAME: full_curated_v1_final_clean_split
TRAIN_MAX_IMAGES: None
EVAL_MAX_IMAGES_PER_SUBSET: None
EPOCHS_YOLO: 30
EPOCHS_RTDETR: 5
EVAL_OUTPUT_ROOT: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_full_curated_v1


In [2]:
# 2) Verify project and dataset paths
required_paths = [
    PROJECT_ROOT,
    PROJECT_ROOT / "code" / "scripts",
    PROJECT_ROOT / "code" / "src",
    PROJECT_ROOT / "configs",
    PROJECT_ROOT / "requirements-colab.txt",
    DATASET_ROOT / "images",
    DATASET_ROOT / "labels",
    DATASET_ROOT / "metadata.csv",
    PROJECT_ROOT / BASE_EVAL_CONFIG,
]

print("Path checks:")
missing = []
for p in required_paths:
    ok = p.exists()
    print(f"{ok!s:5}  {p}")
    if not ok:
        missing.append(str(p))

if missing:
    raise FileNotFoundError("Missing required paths:\n" + "\n".join(missing))

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "code" / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "code" / "src"))

print("\nProject and dataset are ready.")


Path checks:
True   /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification
True   /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/scripts
True   /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/src
True   /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/configs
True   /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/requirements-colab.txt
True   /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/synthetic/full_curated_v1/images
True   /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/synthetic/full_curated_v1/labels
True   /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/synthetic/full_curated_v1/metadata.csv
True   /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/configs/eval_colab_full

In [3]:
# 3) Optional cleanup of old outputs
# RESET_MODEL_OUTPUTS=False by default, because training takes time.
# If you set it to True, this removes all matching Ultralytics runs, including -2/-3 suffix folders.
if RESET_MODEL_OUTPUTS:
    cleanup_patterns = [
        (PROJECT_ROOT / "outputs/training/yolo", f"{YOLO_RUN_NAME}*"),
        (PROJECT_ROOT / "outputs/training/rtdetr", f"{RTDETR_RUN_NAME}*"),
    ]
    for root, pattern in cleanup_patterns:
        for p in root.glob(pattern):
            print("Removing training run:", p)
            shutil.rmtree(p, ignore_errors=True)

if RESET_PREDICTIONS:
    pred_dir = PROJECT_ROOT / "outputs/evaluation/colab_full_curated_v1/predictions"
    if pred_dir.exists():
        for p in pred_dir.glob("*_predictions.csv"):
            print("Removing prediction:", p)
            p.unlink()

print("Cleanup step complete.")

Cleanup step complete.


In [4]:
# 4) Install dependencies + GroundingDINO
# This cell can take several minutes.
os.chdir(PROJECT_ROOT)

run_command([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements-colab.txt")])
run_command([sys.executable, "-m", "pip", "install", "-q", "transformers==4.41.2", "tokenizers<0.20"])

(PROJECT_ROOT / "external").mkdir(exist_ok=True)
(PROJECT_ROOT / "weights/groundingdino").mkdir(parents=True, exist_ok=True)

gdino_dir = PROJECT_ROOT / "external/GroundingDINO"
if not gdino_dir.is_dir():
    run_command(["git", "clone", "https://github.com/IDEA-Research/GroundingDINO.git", str(gdino_dir)])

ckpt = PROJECT_ROOT / "weights/groundingdino/groundingdino_swint_ogc.pth"
if not ckpt.is_file():
    run_command([
        "wget", "-q", "-O", str(ckpt),
        "https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth",
    ])

# Patch for newer torch/CUDA builds when needed.
cuda_file = gdino_dir / "groundingdino/models/GroundingDINO/csrc/MsDeformAttn/ms_deform_attn_cuda.cu"
if cuda_file.exists():
    text = cuda_file.read_text()
    text = text.replace("value.type()", "value.scalar_type()")
    text = text.replace("value.scalar_type().is_cuda()", "value.is_cuda()")
    cuda_file.write_text(text)

run_command([sys.executable, "-m", "pip", "install", "--no-build-isolation", "-e", str(gdino_dir)])

# Verify pinned transformers version
import transformers
print("transformers:", transformers.__version__)



Running:
/usr/bin/python3 -m pip install -q -r '/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/requirements-colab.txt'

===== STDOUT =====
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.2/280.2 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 12.3 MB/s eta 0:00:00


===== STDERR =====


Return code: 0

Running:
/usr/bin/python3 -m pip install -q transformers==4.41.2 tokenizers<0.20

===== STDOUT =====
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9

In [5]:
# 5) Verify imports
os.chdir(PROJECT_ROOT)

gdino_path = str(PROJECT_ROOT / "external/GroundingDINO")
src_path = str(PROJECT_ROOT / "code" / "src")
for p in [gdino_path, src_path]:
    if p not in sys.path:
        sys.path.insert(0, p)

import groundingdino
print("GroundingDINO import OK")

from ultralytics import YOLO
print("Ultralytics import OK")


GroundingDINO import OK
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics import OK


In [6]:
# 6) Create final-clean train / val / test split and Ultralytics dataset.yaml
# Training split contains positives + hard negatives + mixed.
# Evaluation will later be done separately on test_drone_positive, test_hard_negative, test_mixed.

import pandas as pd
import random
from pathlib import Path

metadata_path = DATASET_ROOT / "metadata.csv"
metadata_df = pd.read_csv(metadata_path)

required_cols = {"image_id", "subset"}
missing_cols = required_cols - set(metadata_df.columns)
if missing_cols:
    raise ValueError(f"metadata.csv missing required columns: {missing_cols}")

split_root = PROJECT_ROOT / "data/training" / SPLIT_NAME
split_root.mkdir(parents=True, exist_ok=True)

# Optional cap for quick debugging, stratified by subset.
work_df = metadata_df.copy()
if TRAIN_MAX_IMAGES is not None and TRAIN_MAX_IMAGES < len(work_df):
    parts = []
    per_subset = max(1, TRAIN_MAX_IMAGES // work_df["subset"].nunique())
    for subset, g in work_df.groupby("subset"):
        g = g.sample(frac=1.0, random_state=42)
        parts.append(g.head(per_subset))
    work_df = pd.concat(parts, ignore_index=True)
    if len(work_df) > TRAIN_MAX_IMAGES:
        work_df = work_df.sample(n=TRAIN_MAX_IMAGES, random_state=42).reset_index(drop=True)

# Deterministic split inside every subset.
split_rows = []
rng = random.Random(42)

for subset, g in work_df.groupby("subset"):
    ids = list(g.index)
    rng.shuffle(ids)

    n = len(ids)
    n_train = int(round(n * TRAIN_RATIO))
    n_val = int(round(n * VAL_RATIO))

    train_ids = set(ids[:n_train])
    val_ids = set(ids[n_train:n_train + n_val])

    for idx in ids:
        row = work_df.loc[idx].copy()
        if idx in train_ids:
            row["split_final_clean"] = "train"
        elif idx in val_ids:
            row["split_final_clean"] = "val"
        else:
            row["split_final_clean"] = "test"
        split_rows.append(row)

split_df = pd.DataFrame(split_rows).reset_index(drop=True)

split_csv = split_root / "split_metadata.csv"
split_df.to_csv(split_csv, index=False)

print("Wrote split metadata:", split_csv)
print("\nSplit counts by subset:")
display(pd.crosstab(split_df["subset"], split_df["split_final_clean"]))

# Create image list files for Ultralytics.
# Labels are inferred from image paths by replacing /images/ with /labels/.
def image_path_for_id(image_id):
    return DATASET_ROOT / "images" / str(image_id)

for split_name in ["train", "val", "test"]:
    rows = split_df[split_df["split_final_clean"] == split_name]
    list_path = split_root / f"{split_name}_images.txt"
    with open(list_path, "w", encoding="utf-8") as f:
        for image_id in rows["image_id"].astype(str):
            p = image_path_for_id(image_id)
            if not p.exists():
                raise FileNotFoundError(f"Missing image: {p}")
            f.write(str(p) + "\n")
    print(f"{split_name}: {len(rows)} images -> {list_path}")

# Dataset YAML for Ultralytics training.
DATA_YAML = split_root / "dataset.yaml"
yaml_text = f"""# Auto-generated final clean split
path: "{PROJECT_ROOT}"
train: "{split_root / 'train_images.txt'}"
val: "{split_root / 'val_images.txt'}"
test: "{split_root / 'test_images.txt'}"
nc: 1
names:
  0: drone
"""
DATA_YAML.write_text(yaml_text, encoding="utf-8")

print("\nDATA_YAML:", DATA_YAML)
print(DATA_YAML.read_text())

# Basic label sanity.
for split_name in ["train", "val", "test"]:
    rows = split_df[split_df["split_final_clean"] == split_name]
    non_empty = 0
    missing = 0
    for image_id in rows["image_id"].astype(str):
        label_path = DATASET_ROOT / "labels" / Path(image_id).with_suffix(".txt").name
        if not label_path.exists():
            missing += 1
        elif label_path.read_text(encoding="utf-8").strip():
            non_empty += 1
    print(f"{split_name}: rows={len(rows)}, non_empty_drone_labels={non_empty}, missing_labels={missing}")


Wrote split metadata: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/training/full_curated_v1_final_clean_split/split_metadata.csv

Split counts by subset:


split_final_clean,test,train,val
subset,,,
synthetic_distractor_only,180,840,180
synthetic_drone_plus_distractor,180,840,180
synthetic_drone_positive,180,840,180


train: 2520 images -> /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/training/full_curated_v1_final_clean_split/train_images.txt
val: 540 images -> /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/training/full_curated_v1_final_clean_split/val_images.txt
test: 540 images -> /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/training/full_curated_v1_final_clean_split/test_images.txt

DATA_YAML: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/training/full_curated_v1_final_clean_split/dataset.yaml
# Auto-generated final clean split
path: "/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification"
train: "/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/training/full_curated_v1_final_clean_split/train_images.txt"
val: "/content/drive/MyDrive/Synthetic Stress Testing for Skyborn

In [7]:
# 7) Create held-out evaluation subset datasets and generated eval configs
# Each eval subset contains only TEST rows from one condition:
# - test_drone_positive
# - test_hard_negative
# - test_mixed

import yaml
import pandas as pd
from pathlib import Path
import shutil
import os

split_root = PROJECT_ROOT / "data/training" / SPLIT_NAME
split_df = pd.read_csv(split_root / "split_metadata.csv")

eval_dataset_root = PROJECT_ROOT / "data/evaluation/final_clean_full_curated_v1"
eval_dataset_root.mkdir(parents=True, exist_ok=True)
EVAL_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
EVAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def copy_or_link(src, dst):
    """Try symlink first; fallback to copy. Google Drive sometimes behaves better with copies."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        return
    try:
        os.symlink(src, dst)
    except Exception:
        shutil.copy2(src, dst)

def replace_strings(obj, replacements):
    """Recursively replace string fragments in a loaded YAML object."""
    if isinstance(obj, dict):
        return {k: replace_strings(v, replacements) for k, v in obj.items()}
    if isinstance(obj, list):
        return [replace_strings(v, replacements) for v in obj]
    if isinstance(obj, str):
        s = obj
        for old, new in replacements.items():
            s = s.replace(old, new)
        return s
    return obj

def make_eval_subset_dataset(eval_name, subset_value):
    rows = split_df[
        (split_df["split_final_clean"] == "test") &
        (split_df["subset"] == subset_value)
    ].copy()

    if EVAL_MAX_IMAGES_PER_SUBSET is not None:
        rows = rows.head(EVAL_MAX_IMAGES_PER_SUBSET).copy()

    subset_root = eval_dataset_root / eval_name
    images_dir = subset_root / "images"
    labels_dir = subset_root / "labels"
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)

    for image_id in rows["image_id"].astype(str):
        src_img = DATASET_ROOT / "images" / image_id
        src_lbl = DATASET_ROOT / "labels" / Path(image_id).with_suffix(".txt").name
        dst_img = images_dir / image_id
        dst_lbl = labels_dir / Path(image_id).with_suffix(".txt").name

        if not src_img.exists():
            raise FileNotFoundError(src_img)
        copy_or_link(src_img, dst_img)

        # Hard negatives should have empty labels; if label is missing, create empty file.
        if src_lbl.exists():
            copy_or_link(src_lbl, dst_lbl)
        else:
            dst_lbl.write_text("", encoding="utf-8")

    rows.to_csv(subset_root / "metadata.csv", index=False)
    return subset_root, rows

def make_eval_config(eval_name, subset_root):
    base_path = PROJECT_ROOT / BASE_EVAL_CONFIG
    cfg = yaml.safe_load(base_path.read_text(encoding="utf-8"))

    # Keep paths relative to PROJECT_ROOT when possible, because scripts receive --project-root.
    rel_subset_root = subset_root.relative_to(PROJECT_ROOT).as_posix()
    rel_output_root = (EVAL_OUTPUT_ROOT / eval_name).relative_to(PROJECT_ROOT).as_posix()

    replacements = {
        "data/synthetic/full_curated_v1": rel_subset_root,
        "outputs/evaluation/colab_full_curated_v1": rel_output_root,
        "outputs/evaluation/full_curated_v1": rel_output_root,
    }
    cfg = replace_strings(cfg, replacements)

    # Patch common schema keys explicitly if present.
    if isinstance(cfg, dict):
        for key in ["dataset_root", "data_root", "root"]:
            if key in cfg and isinstance(cfg[key], str):
                cfg[key] = rel_subset_root
        if "dataset" in cfg and isinstance(cfg["dataset"], dict):
            for key in ["root", "dataset_root", "data_root"]:
                if key in cfg["dataset"]:
                    cfg["dataset"][key] = rel_subset_root
            for key in ["metadata", "metadata_path"]:
                if key in cfg["dataset"]:
                    cfg["dataset"][key] = f"{rel_subset_root}/metadata.csv"
        for key in ["output_dir", "output_root"]:
            if key in cfg:
                cfg[key] = rel_output_root
        if "evaluation" in cfg and isinstance(cfg["evaluation"], dict):
            for key in ["output_dir", "output_root"]:
                if key in cfg["evaluation"]:
                    cfg["evaluation"][key] = rel_output_root

    out_config = EVAL_CONFIG_DIR / f"eval_{eval_name}.yaml"
    out_config.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
    return out_config

EVAL_JOBS = {}

for eval_name, subset_value in EVAL_SUBSETS.items():
    subset_root, rows = make_eval_subset_dataset(eval_name, subset_value)
    cfg_path = make_eval_config(eval_name, subset_root)
    EVAL_JOBS[eval_name] = {
        "subset": subset_value,
        "dataset_root": subset_root,
        "config": cfg_path,
        "output_root": EVAL_OUTPUT_ROOT / eval_name,
        "n_images": len(rows),
        "n_target_present": int(rows["target_present"].astype(bool).sum()) if "target_present" in rows.columns else None,
    }

print("Created evaluation jobs:")
for name, job in EVAL_JOBS.items():
    print("\n", name)
    for k, v in job.items():
        print(" ", k, ":", v)

print("\nTest counts:")
display(pd.DataFrame([
    {"eval_subset": name, **{k: str(v) for k, v in job.items() if k != "config"}}
    for name, job in EVAL_JOBS.items()
]))

print("\nExample generated config:")
first_cfg = next(iter(EVAL_JOBS.values()))["config"]
print(first_cfg)
print(first_cfg.read_text()[:2000])


Created evaluation jobs:

 test_drone_positive
  subset : synthetic_drone_positive
  dataset_root : /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/evaluation/final_clean_full_curated_v1/test_drone_positive
  config : /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/configs/generated_eval_subsets/eval_test_drone_positive.yaml
  output_root : /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_full_curated_v1/test_drone_positive
  n_images : 180
  n_target_present : 180

 test_hard_negative
  subset : synthetic_distractor_only
  dataset_root : /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/evaluation/final_clean_full_curated_v1/test_hard_negative
  config : /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/configs/generated_eval_subsets/eval_test_hard_negative.yaml
  output_root : /con

,eval_subset,subset,dataset_root,output_root,n_images,n_target_present
0,test_drone_positive,synthetic_drone_positive,/content/drive/MyDrive/Synthetic Stress Testin...,/content/drive/MyDrive/Synthetic Stress Testin...,180,180
1,test_hard_negative,synthetic_distractor_only,/content/drive/MyDrive/Synthetic Stress Testin...,/content/drive/MyDrive/Synthetic Stress Testin...,180,0
2,test_mixed,synthetic_drone_plus_distractor,/content/drive/MyDrive/Synthetic Stress Testin...,/content/drive/MyDrive/Synthetic Stress Testin...,180,180



Example generated config:
/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/configs/generated_eval_subsets/eval_test_drone_positive.yaml
dataset:
  name: full_curated_v1
  root: data/evaluation/final_clean_full_curated_v1/test_drone_positive
  images_dir: data/evaluation/final_clean_full_curated_v1/test_drone_positive/images
  labels_dir: data/evaluation/final_clean_full_curated_v1/test_drone_positive/labels
  metadata_csv: data/evaluation/final_clean_full_curated_v1/test_drone_positive/metadata.csv
  image_width: 640
  image_height: 640
evaluation:
  iou_thresholds:
  - 0.25
  - 0.5
  confidence_thresholds:
  - 0.05
  - 0.1
  - 0.25
  - 0.5
  primary_iou_threshold: 0.5
  tiny_object_iou_threshold: 0.25
  target_class_name: drone
  drone_class_id: 0
  drone_like_classes:
  - drone
  - quadcopter
  - unmanned aerial vehicle
outputs:
  root: outputs/evaluation/final_clean_full_curated_v1/test_drone_positive
grounding_dino:
  repo_dir: external/GroundingDI

In [8]:
# 8) Fine-tune YOLO11n
if RUN_YOLO_TRAINING:
    cmd = [
        sys.executable,
        "code/code/scripts/train/train_yolo.py",
        "--data", str(DATA_YAML),
        "--weights", "yolo11n.pt",
        "--epochs", str(EPOCHS_YOLO),
        "--imgsz", str(IMGSZ),
        "--batch", str(BATCH_YOLO),
        "--device", "0",
        "--name", YOLO_RUN_NAME,
        "--workers", "2",
    ]
    run_command(cmd)
else:
    print("Skipping YOLO training.")



Running:
/usr/bin/python3 scripts/train/train_yolo.py --data '/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/training/full_curated_v1_final_clean_split/dataset.yaml' --weights yolo11n.pt --epochs 30 --imgsz 640 --batch 16 --device 0 --name yolo11n_drone_colab --workers 2

===== STDOUT =====
Training YOLO from yolo11n.pt
  data=/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/training/full_curated_v1_final_clean_split/dataset.yaml
  epochs=30 imgsz=640 batch=16 device=0
Ultralytics 8.4.80 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone 

In [9]:
# 9) Inspect YOLO training results
import pandas as pd

yolo_root = PROJECT_ROOT / "outputs/training/yolo"
stable_yolo = yolo_root / YOLO_RUN_NAME
latest_yolo = get_latest_training_run(yolo_root, f"{YOLO_RUN_NAME}*")

print("Stable YOLO folder:", stable_yolo)
print("Latest YOLO folder:", latest_yolo)

for label, run_dir in [("stable", stable_yolo), ("latest", latest_yolo)]:
    if run_dir is None:
        continue
    weights_best = run_dir / "weights/best.pt"
    results_csv = run_dir / "results.csv"
    print(f"\n{label.upper()} YOLO best weights:", weights_best, "exists:", weights_best.exists())
    print(f"{label.upper()} YOLO results.csv:", results_csv, "exists:", results_csv.exists())
    if results_csv.exists():
        yolo_results = pd.read_csv(results_csv)
        print(f"{label.upper()} YOLO epochs:", len(yolo_results))
        display(yolo_results.tail())

Stable YOLO folder: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/yolo/yolo11n_drone_colab
Latest YOLO folder: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/yolo/yolo11n_drone_colab-7

STABLE YOLO best weights: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/yolo/yolo11n_drone_colab/weights/best.pt exists: True
STABLE YOLO results.csv: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/yolo/yolo11n_drone_colab/results.csv exists: True
STABLE YOLO epochs: 30


,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
25,26,390.302,1.87272,1.60086,1.25133,0.50000,0.56552,0.49535,0.20935,1.91642,1.58180,1.30507,0.000350,0.000350,0.000350
26,27,403.436,1.85931,1.63125,1.23196,0.47561,0.55045,0.50841,0.22807,1.87148,1.55491,1.27199,0.000284,0.000284,0.000284
27,28,416.729,1.84009,1.63754,1.22470,0.51329,0.53821,0.51853,0.23432,1.83300,1.50185,1.25394,0.000218,0.000218,0.000218
28,29,429.837,1.81727,1.59613,1.23711,0.53052,0.57241,0.55470,0.24955,1.82812,1.55066,1.24967,0.000152,0.000152,0.000152
29,30,443.319,1.79738,1.55943,1.22804,0.51046,0.55374,0.53555,0.24151,1.83378,1.52233,1.24812,0.000086,0.000086,0.000086



LATEST YOLO best weights: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/yolo/yolo11n_drone_colab-7/weights/best.pt exists: True
LATEST YOLO results.csv: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/yolo/yolo11n_drone_colab-7/results.csv exists: True
LATEST YOLO epochs: 30


,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
25,26,910.551,1.81570,1.51449,1.19726,0.66619,0.52110,0.53663,0.25263,1.82409,1.47756,1.18572,0.000350,0.000350,0.000350
26,27,942.886,1.84728,1.52717,1.20611,0.67462,0.50683,0.53182,0.24989,1.83825,1.49472,1.19969,0.000284,0.000284,0.000284
27,28,974.522,1.79784,1.53891,1.20076,0.69637,0.51944,0.53859,0.26002,1.81157,1.43168,1.19338,0.000218,0.000218,0.000218
28,29,1006.810,1.79608,1.52693,1.20577,0.72903,0.52316,0.56563,0.27433,1.77564,1.44200,1.17583,0.000152,0.000152,0.000152
29,30,1038.380,1.79054,1.50184,1.18813,0.73865,0.51031,0.56941,0.27238,1.76650,1.44387,1.17692,0.000086,0.000086,0.000086


In [ ]:
# 10) Fine-tune RT-DETR-L
# RT-DETR is heavier than YOLO. You can set RUN_RTDETR_TRAINING=False above if the session is short.
if RUN_RTDETR_TRAINING:
    cmd = [
        sys.executable,
        "code/code/scripts/train/train_rtdetr.py",
        "--data", str(DATA_YAML),
        "--weights", "rtdetr-l.pt",
        "--epochs", str(EPOCHS_RTDETR),
        "--imgsz", str(IMGSZ),
        "--batch", str(BATCH_RTDETR),
        "--device", "0",
        "--name", RTDETR_RUN_NAME,
        "--workers", "2",
    ]
    run_command(cmd)
else:
    print("Skipping RT-DETR training.")



Running:
/usr/bin/python3 scripts/train/train_rtdetr.py --data '/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/training/full_curated_v1_final_clean_split/dataset.yaml' --weights rtdetr-l.pt --epochs 5 --imgsz 640 --batch 2 --device 0 --name rtdetr_l_drone_colab --workers 2


In [ ]:
# 11) Sync latest Ultralytics runs to stable folders expected by eval config
# This fixes the common Colab/Ultralytics issue where new training runs are saved as -2, -3, ...
# while the eval config still points to the original stable folder name.

if RUN_YOLO_TRAINING or RUN_YOLO_INFERENCE:
    sync_latest_run_to_stable(
        PROJECT_ROOT / "outputs/training/yolo",
        f"{YOLO_RUN_NAME}*",
        YOLO_RUN_NAME,
    )

if RUN_RTDETR_TRAINING or RUN_RTDETR_INFERENCE:
    sync_latest_run_to_stable(
        PROJECT_ROOT / "outputs/training/rtdetr",
        f"{RTDETR_RUN_NAME}*",
        RTDETR_RUN_NAME,
    )

# Show final stable folders and epoch counts
for label, root, run_name in [
    ("YOLO", PROJECT_ROOT / "outputs/training/yolo", YOLO_RUN_NAME),
    ("RT-DETR", PROJECT_ROOT / "outputs/training/rtdetr", RTDETR_RUN_NAME),
]:
    stable = root / run_name
    results_csv = stable / "results.csv"
    weights = stable / "weights/best.pt"
    print(f"\n{label} stable folder: {stable}")
    print("  weights exists:", weights.exists())
    print("  results exists:", results_csv.exists())
    if results_csv.exists():
        df = pd.read_csv(results_csv)
        print("  epochs:", len(df))
        display(df.tail())

Backed up old stable folder to: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/yolo/yolo11n_drone_colab_old_before_latest_sync
Synced latest run to stable folder:
  latest: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/yolo/yolo11n_drone_colab-5
  stable: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/yolo/yolo11n_drone_colab
Backed up old stable folder to: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/rtdetr/rtdetr_l_drone_colab_old_before_latest_sync
Synced latest run to stable folder:
  latest: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/rtdetr/rtdetr_l_drone_colab-5
  stable: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/rtdetr/rtdetr_l_drone_colab

YOLO stable folder: 

,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
25,26,390.302,1.87272,1.60086,1.25133,0.50000,0.56552,0.49535,0.20935,1.91642,1.58180,1.30507,0.000350,0.000350,0.000350
26,27,403.436,1.85931,1.63125,1.23196,0.47561,0.55045,0.50841,0.22807,1.87148,1.55491,1.27199,0.000284,0.000284,0.000284
27,28,416.729,1.84009,1.63754,1.22470,0.51329,0.53821,0.51853,0.23432,1.83300,1.50185,1.25394,0.000218,0.000218,0.000218
28,29,429.837,1.81727,1.59613,1.23711,0.53052,0.57241,0.55470,0.24955,1.82812,1.55066,1.24967,0.000152,0.000152,0.000152
29,30,443.319,1.79738,1.55943,1.22804,0.51046,0.55374,0.53555,0.24151,1.83378,1.52233,1.24812,0.000086,0.000086,0.000086



RT-DETR stable folder: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/rtdetr/rtdetr_l_drone_colab
  weights exists: True
  results exists: True
  epochs: 5


,epoch,time,train/giou_loss,train/cls_loss,train/l1_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/giou_loss,val/cls_loss,val/l1_loss,lr/pg0,lr/pg1,lr/pg2
0,1,234.121,1.92744,1.91841,0.50162,0.46122,0.41379,0.37740,0.14625,1.15531,0.49085,0.26834,0.000665,0.000665,0.000665
1,2,453.364,1.41728,0.48042,0.25409,0.50084,0.44138,0.37412,0.14063,1.08774,0.50872,0.21163,0.001068,0.001068,0.001068
2,3,672.696,1.28809,0.45589,0.20135,0.58269,0.57241,0.53138,0.22332,0.94069,0.47573,0.17678,0.001207,0.001207,0.001207
3,4,894.668,1.21509,0.47262,0.19161,0.68301,0.53496,0.54167,0.20975,1.11191,0.42655,0.24454,0.000812,0.000812,0.000812
4,5,1114.350,1.21382,0.43582,0.17647,0.73734,0.48276,0.53438,0.20725,0.97615,0.47758,0.16406,0.000416,0.000416,0.000416


: 

In [ ]:
# 12) Inference on each held-out evaluation subset
# Runs each enabled detector on:
# - test_drone_positive
# - test_hard_negative
# - test_mixed

models_to_run = []
if RUN_GDINO:
    models_to_run.append("grounding_dino")
if RUN_YOLO_INFERENCE:
    models_to_run.append("yolo11n_drone_finetuned")
if RUN_RTDETR_INFERENCE:
    models_to_run.append("rtdetr_l_drone_finetuned")

for eval_name, job in EVAL_JOBS.items():
    print("\n" + "#" * 100)
    print("EVAL SUBSET:", eval_name, "| source subset:", job["subset"], "| images:", job["n_images"])
    print("#" * 100)

    # Delete stale predictions for this eval subset.
    pred_dir = job["output_root"] / "predictions"
    pred_dir.mkdir(parents=True, exist_ok=True)
    if RESET_PREDICTIONS:
        for model_name in models_to_run:
            p = pred_dir / f"{model_name}_predictions.csv"
            if p.exists():
                p.unlink()
                print("Deleted stale prediction file:", p)

    base_cmd = [
        sys.executable,
        "code/code/scripts/eval/run_detectors.py",
        "--config", str(job["config"]),
        "--project-root", str(PROJECT_ROOT),
    ]

    if RUN_GDINO:
        run_command(base_cmd + ["--models", "grounding_dino"], check=True)

    if RUN_YOLO_INFERENCE:
        run_command(base_cmd + ["--models", "yolo11n_drone_finetuned"], check=False)

    if RUN_RTDETR_INFERENCE:
        run_command(base_cmd + ["--models", "rtdetr_l_drone_finetuned"], check=False)


Deleted stale prediction file: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/colab_full_curated_v1/predictions/grounding_dino_predictions.csv
Deleted stale prediction file: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/colab_full_curated_v1/predictions/yolo11n_drone_finetuned_predictions.csv
Deleted stale prediction file: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/colab_full_curated_v1/predictions/rtdetr_l_drone_finetuned_predictions.csv

Running:
/usr/bin/python3 scripts/eval/run_detectors.py --config configs/eval_colab_full_curated_v1.yaml --project-root '/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification' --models grounding_dino --max-images 300

===== STDOUT =====
Preflight OK: 3601 images, metadata=/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/s

: 

In [ ]:
# 13) Inspect prediction CSVs by evaluation subset
import pandas as pd

for eval_name, job in EVAL_JOBS.items():
    pred_dir = job["output_root"] / "predictions"
    print("\n" + "#" * 100)
    print("EVAL SUBSET:", eval_name)
    print("Prediction directory:", pred_dir)
    print("#" * 100)

    if not pred_dir.exists():
        print("Prediction directory does not exist.")
        continue

    for csv_path in sorted(pred_dir.glob("*_predictions.csv")):
        df = pd.read_csv(csv_path)
        print("\n" + "=" * 80)
        print(csv_path.name)
        print("shape:", df.shape)
        if "model_name" in df.columns and len(df):
            print("model_name:", df["model_name"].unique())
        if "confidence" in df.columns and len(df):
            print("confidence min/mean/max:", df["confidence"].min(), df["confidence"].mean(), df["confidence"].max())
        if "image_id" in df.columns and len(df):
            print("first image_ids:", df["image_id"].head().tolist())
        display(df.head())


Prediction directory: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/colab_full_curated_v1/predictions

grounding_dino_predictions.csv
shape: (3425, 14)
columns: ['image_id', 'image_path', 'model_name', 'pred_class_id', 'pred_class_name', 'confidence', 'x1', 'y1', 'x2', 'y2', 'width', 'height', 'prompt', 'raw_label']


,image_id,image_path,model_name,pred_class_id,pred_class_name,confidence,x1,y1,x2,y2,width,height,prompt,raw_label
0,img_000001.png,/content/drive/MyDrive/Synthetic Stress Testin...,grounding_dino,0,drone,0.244451,0.05,34.48,536.18,639.00,536.13,604.52,drone,drone
1,img_000001.png,/content/drive/MyDrive/Synthetic Stress Testin...,grounding_dino,0,drone,0.210452,0.23,35.14,537.39,639.14,537.16,604.00,quadcopter,quadcopter
2,img_000001.png,/content/drive/MyDrive/Synthetic Stress Testin...,grounding_dino,0,drone,0.200748,0.14,414.84,536.59,639.87,536.45,225.03,quadcopter,##coer
3,img_000001.png,/content/drive/MyDrive/Synthetic Stress Testin...,grounding_dino,0,drone,0.314957,0.13,34.25,538.53,638.34,538.40,604.09,unmanned aerial vehicle,unmanned aerial vehicle
4,img_000001.png,/content/drive/MyDrive/Synthetic Stress Testin...,grounding_dino,0,drone,0.222389,0.19,212.03,374.57,482.66,374.38,270.63,unmanned aerial vehicle,unmanned aerial vehicle



rtdetr_l_drone_finetuned_predictions.csv
shape: (5659, 14)
columns: ['image_id', 'image_path', 'model_name', 'pred_class_id', 'pred_class_name', 'confidence', 'x1', 'y1', 'x2', 'y2', 'width', 'height', 'prompt', 'raw_label']


,image_id,image_path,model_name,pred_class_id,pred_class_name,confidence,x1,y1,x2,y2,width,height,prompt,raw_label
0,img_000001.png,/content/drive/MyDrive/Synthetic Stress Testin...,rtdetr_l_drone_finetuned,0,drone,0.358268,307.73,139.48,317.96,147.52,10.24,8.05,NaN,drone
1,img_000001.png,/content/drive/MyDrive/Synthetic Stress Testin...,rtdetr_l_drone_finetuned,0,drone,0.251808,309.95,140.78,317.12,146.18,7.18,5.40,NaN,drone
2,img_000001.png,/content/drive/MyDrive/Synthetic Stress Testin...,rtdetr_l_drone_finetuned,0,drone,0.189414,175.63,174.12,202.91,191.08,27.28,16.96,NaN,drone
3,img_000001.png,/content/drive/MyDrive/Synthetic Stress Testin...,rtdetr_l_drone_finetuned,0,drone,0.185377,117.80,349.74,133.08,359.71,15.27,9.97,NaN,drone
4,img_000001.png,/content/drive/MyDrive/Synthetic Stress Testin...,rtdetr_l_drone_finetuned,0,drone,0.086730,253.33,307.19,272.33,322.09,19.00,14.90,NaN,drone



yolo11n_drone_finetuned_predictions.csv
shape: (392, 14)
columns: ['image_id', 'image_path', 'model_name', 'pred_class_id', 'pred_class_name', 'confidence', 'x1', 'y1', 'x2', 'y2', 'width', 'height', 'prompt', 'raw_label']


,image_id,image_path,model_name,pred_class_id,pred_class_name,confidence,x1,y1,x2,y2,width,height,prompt,raw_label
0,img_000001.png,/content/drive/MyDrive/Synthetic Stress Testin...,yolo11n_drone_finetuned,0,drone,0.078592,309.73,139.90,318.11,146.26,8.38,6.36,NaN,drone
1,img_000001.png,/content/drive/MyDrive/Synthetic Stress Testin...,yolo11n_drone_finetuned,0,drone,0.071272,309.31,140.59,318.28,148.07,8.96,7.48,NaN,drone
2,img_000002.png,/content/drive/MyDrive/Synthetic Stress Testin...,yolo11n_drone_finetuned,0,drone,0.564265,161.38,247.19,215.39,280.35,54.01,33.17,NaN,drone
3,img_000002.png,/content/drive/MyDrive/Synthetic Stress Testin...,yolo11n_drone_finetuned,0,drone,0.238872,467.87,303.83,545.18,352.86,77.31,49.03,NaN,drone
4,img_000002.png,/content/drive/MyDrive/Synthetic Stress Testin...,yolo11n_drone_finetuned,0,drone,0.185688,155.77,241.92,219.60,285.74,63.82,43.82,NaN,drone


: 

In [ ]:
# 14) Metrics + prediction QA by held-out evaluation subset
# Writes one combined table with eval_subset, model_name, recall, FP rates/counts, etc.

import pandas as pd
from pathlib import Path

models = [
    "grounding_dino",
    "yolo11n_drone_finetuned",
    "rtdetr_l_drone_finetuned",
]

all_rows = []

for eval_name, job in EVAL_JOBS.items():
    print("\n" + "#" * 100)
    print("METRICS FOR EVAL SUBSET:", eval_name)
    print("#" * 100)

    metadata_df = pd.read_csv(job["dataset_root"] / "metadata.csv")
    n_target_present = int(metadata_df["target_present"].astype(bool).sum()) if "target_present" in metadata_df.columns else 0
    n_distractor_only = int((metadata_df["subset"] == "synthetic_distractor_only").sum()) if "subset" in metadata_df.columns else 0

    pred_dir = job["output_root"] / "predictions"
    metrics_dir = job["output_root"] / "metrics"
    metrics_dir.mkdir(parents=True, exist_ok=True)

    subset_rows = []

    for model in models:
        pred_csv = pred_dir / f"{model}_predictions.csv"
        print("\n" + "=" * 80)
        print("MODEL:", model)
        print("=" * 80)

        if not pred_csv.is_file():
            print("SKIP missing prediction file:", pred_csv)
            continue

        pred_df = pd.read_csv(pred_csv)
        print("prediction rows:", len(pred_df))

        if len(pred_df) == 0:
            print("Empty prediction CSV. Adding zero-detection metrics instead of crashing.")
            for iou_thr in [0.25, 0.50]:
                subset_rows.append({
                    "eval_subset": eval_name,
                    "source_subset": job["subset"],
                    "model_name": model,
                    "model_type": "empty_or_underconfident",
                    "iou_threshold": iou_thr,
                    "n_target_present": n_target_present,
                    "n_distractor_only": n_distractor_only,
                    "recall": 0.0,
                    "tp": 0,
                    "fn": n_target_present,
                    "mean_best_iou": 0.0,
                    "false_positive_image_rate": 0.0,
                    "fp_count": 0,
                    "note": "No predictions at configured inference threshold; likely undertrained or threshold too high.",
                })
            continue

        # Evaluate this model for this subset.
        result = run_command([
            sys.executable,
            "code/code/scripts/eval/evaluate_detector_predictions.py",
            "--config", str(job["config"]),
            "--predictions", str(pred_csv),
            "--project-root", str(PROJECT_ROOT),
        ], check=False)

        if result.returncode == 0:
            summary_path = metrics_dir / "summary_by_model.csv"
            if summary_path.exists():
                summary_df = pd.read_csv(summary_path)
                summary_df["eval_subset"] = eval_name
                summary_df["source_subset"] = job["subset"]
                subset_rows.extend(summary_df.to_dict("records"))
        else:
            print("Evaluation failed for this model; see output above.")
            continue

        # Visualization / prediction QA
        run_command([
            sys.executable,
            "code/code/scripts/eval/visualize_predictions.py",
            "--config", str(job["config"]),
            "--predictions", str(pred_csv),
            "--project-root", str(PROJECT_ROOT),
            "--num-samples", "12",
        ], check=False)

    subset_summary = pd.DataFrame(subset_rows)
    if len(subset_summary):
        subset_summary = subset_summary.drop_duplicates(
            subset=["eval_subset", "model_name", "iou_threshold"],
            keep="last",
        ).sort_values(["eval_subset", "model_name", "iou_threshold"]).reset_index(drop=True)

    subset_summary_path = metrics_dir / "summary_by_model_with_eval_subset.csv"
    subset_summary.to_csv(subset_summary_path, index=False)
    print("\nSubset summary written to:", subset_summary_path)
    display(subset_summary)

    all_rows.extend(subset_summary.to_dict("records"))

combined_summary = pd.DataFrame(all_rows)

if len(combined_summary):
    combined_summary = combined_summary.drop_duplicates(
        subset=["eval_subset", "model_name", "iou_threshold"],
        keep="last",
    ).sort_values(["eval_subset", "model_name", "iou_threshold"]).reset_index(drop=True)

combined_metrics_dir = EVAL_OUTPUT_ROOT / "combined_metrics"
combined_metrics_dir.mkdir(parents=True, exist_ok=True)

combined_path = combined_metrics_dir / "summary_by_model_by_eval_subset_CORRECTED.csv"
combined_summary.to_csv(combined_path, index=False)

print("\nCombined final-clean summary written to:")
print(combined_path)
display(combined_summary)

# Optional pivots for quick comparison.
if len(combined_summary):
    print("\nRecall pivot:")
    display(combined_summary.pivot_table(
        index=["model_name", "iou_threshold"],
        columns="eval_subset",
        values="recall",
        aggfunc="first",
    ))

    print("\nFP count pivot:")
    display(combined_summary.pivot_table(
        index=["model_name", "iou_threshold"],
        columns="eval_subset",
        values="fp_count",
        aggfunc="first",
    ))



MODEL: grounding_dino
prediction rows: 3425

Running:
/usr/bin/python3 scripts/eval/evaluate_detector_predictions.py --config configs/eval_colab_full_curated_v1.yaml --predictions '/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/colab_full_curated_v1/predictions/grounding_dino_predictions.csv' --project-root '/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification'

===== STDOUT =====
Evaluating grounding_dino from /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/colab_full_curated_v1/predictions/grounding_dino_predictions.csv

Wrote summary: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/colab_full_curated_v1/metrics/summary_by_model.csv
Wrote 7 plots to /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/colab_full_curated_v1/plots

Summary by model:
  

,model_name,model_type,iou_threshold,n_target_present,n_distractor_only,recall,tp,fn,mean_best_iou,false_positive_image_rate,fp_count,note
0,grounding_dino,grounding_dino,0.25,212,88,0.523585,111,101,0.333507,1.0,1025,Open-vocabulary drone prompts; boxes count as ...
1,grounding_dino,grounding_dino,0.50,212,88,0.410377,87,125,0.287629,1.0,1025,Open-vocabulary drone prompts; boxes count as ...
2,rtdetr_l_drone_finetuned,ultralytics_rtdetr,0.25,212,88,0.806604,171,41,0.578268,1.0,1562,Fine-tuned / drone-capable checkpoint; drone_l...
3,rtdetr_l_drone_finetuned,ultralytics_rtdetr,0.50,212,88,0.759434,161,51,0.558707,1.0,1562,Fine-tuned / drone-capable checkpoint; drone_l...
4,yolo11n_drone_finetuned,ultralytics_yolo,0.25,159,30,0.886792,141,18,0.642325,1.0,52,Fine-tuned / drone-capable checkpoint; drone_l...
5,yolo11n_drone_finetuned,ultralytics_yolo,0.50,159,30,0.805031,128,31,0.610624,1.0,52,Fine-tuned / drone-capable checkpoint; drone_l...


: 

In [ ]:
# 15) Optional: YOLO confidence diagnostic on one positive test image
# Use this if YOLO predictions are empty or unexpectedly weak.

from pathlib import Path

try:
    from ultralytics import YOLO

    weights = PROJECT_ROOT / "outputs/training/yolo" / YOLO_RUN_NAME / "weights/best.pt"

    # Pick first test positive image if available.
    pos_root = EVAL_JOBS["test_drone_positive"]["dataset_root"]
    pos_meta = pd.read_csv(pos_root / "metadata.csv")
    sample_image_id = pos_meta["image_id"].astype(str).iloc[0]
    image = pos_root / "images" / sample_image_id

    print("weights exists:", weights.exists(), weights)
    print("sample image exists:", image.exists(), image)

    if weights.exists() and image.exists():
        model = YOLO(str(weights))
        for conf in [0.25, 0.10, 0.05, 0.02, 0.01, 0.005, 0.001]:
            results = model.predict(
                source=str(image),
                conf=conf,
                imgsz=640,
                device=0,
                verbose=False,
                max_det=300,
            )
            n_boxes = len(results[0].boxes)
            if n_boxes:
                confs = results[0].boxes.conf.cpu().numpy()
                print(f"conf={conf}: boxes={n_boxes}, max_conf={confs.max():.5f}, min_conf={confs.min():.5f}")
            else:
                print(f"conf={conf}: boxes=0")

except Exception as e:
    print("YOLO confidence diagnostic failed:", repr(e))


weights exists: True /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/yolo/yolo11n_drone_colab/weights/best.pt
sample image exists: True /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/synthetic/full_curated_v1/images/img_000002.png
conf=0.25: boxes=1, max_conf=0.56427, min_conf=0.56427
conf=0.1: boxes=3, max_conf=0.56427, min_conf=0.18569
conf=0.05: boxes=3, max_conf=0.56427, min_conf=0.18569
conf=0.02: boxes=4, max_conf=0.56427, min_conf=0.04292
conf=0.01: boxes=4, max_conf=0.56427, min_conf=0.04292
conf=0.005: boxes=5, max_conf=0.56427, min_conf=0.00980
conf=0.001: boxes=12, max_conf=0.56427, min_conf=0.00104


: 

In [ ]:
# 16) Save final-clean results zip to Drive
results_dir = PROJECT_ROOT / "drone_colab_results"
results_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_base = results_dir / f"drone_colab_final_clean_results_{timestamp}"

archive_path = shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=str(EVAL_OUTPUT_ROOT),
)

print("Saved results archive:")
print(archive_path)


Saved results zip:
/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/drone_colab_results/drone_colab_results_medium_20260626_141738.zip


: 

## Notes

This notebook implements the final clean subset evaluation protocol:

- Train YOLO/RT-DETR on the **train split containing all three conditions**:
  - `synthetic_drone_positive`
  - `synthetic_distractor_only`
  - `synthetic_drone_plus_distractor`

- Evaluate GroundingDINO, YOLO, and RT-DETR separately on held-out test subsets:
  - `test_drone_positive`
  - `test_hard_negative`
  - `test_mixed`

- The main final table is saved to:

`outputs/evaluation/final_clean_full_curated_v1/combined_metrics/summary_by_model_by_eval_subset_CORRECTED.csv`

Scientific interpretation:

- For `test_drone_positive` and `test_mixed`, recall and IoU metrics are meaningful.
- For `test_hard_negative`, recall is not meaningful because no drone exists. Focus on false-positive image rate and FP count.
- Hard negatives are included in training as negative examples, not as a standalone detector-training dataset.
